# 🔍 HuggingFace Model Recommender

**Detects your hardware → Fetches text-generation models → Scores them → Recommends what will actually run on your machine.**

Scoring dimensions:
- **Fit** (40%): Will it fit in your VRAM/RAM?
- **Quality** (25%): Downloads, likes, community trust
- **Speed** (25%): How fast will inference be on your hardware?
- **Context** (10%): How large is the context window?

> Run cells top-to-bottom. Cell 1 detects hardware, Cell 2 fetches models, Cells 3-5 score and display results.

In [7]:
# ============================================================
# CELL 1: Hardware Detection
# ============================================================
import platform, os

try:
    import psutil
except ImportError:
    import pip; pip.main(["install", "psutil"]); import psutil

try:
    import torch
except ImportError:
    torch = None

# ---------- CPU ----------
cpu_model = platform.processor() or platform.machine()
cpu_cores_logical = os.cpu_count()
cpu_cores_physical = psutil.cpu_count(logical=False)

# ---------- RAM ----------
mem = psutil.virtual_memory()
ram_total_gb = round(mem.total / (1024**3), 1)
ram_available_gb = round(mem.available / (1024**3), 1)

# ---------- GPU / Accelerator ----------
device_type = "cpu"
gpu_name = None
vram_total_gb = 0.0
vram_available_gb = 0.0

if torch is not None and torch.cuda.is_available():
    device_type = "cuda"
    gpu_name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    vram_total_gb = round(props.total_mem / (1024**3), 1)
    vram_free, _ = torch.cuda.mem_get_info(0)
    vram_available_gb = round(vram_free / (1024**3), 1)
elif torch is not None and hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device_type = "mps"
    gpu_name = "Apple Silicon (MPS)"
    vram_total_gb = round(ram_total_gb * 0.70, 1)
    vram_available_gb = round(ram_available_gb * 0.70, 1)
else:
    vram_total_gb = round(ram_total_gb * 0.80, 1)
    vram_available_gb = round(ram_available_gb * 0.80, 1)

# Build hardware dict (used by scoring engine later)
HARDWARE = {
    "cpu_model": cpu_model,
    "cpu_cores_logical": cpu_cores_logical,
    "cpu_cores_physical": cpu_cores_physical,
    "ram_total_gb": ram_total_gb,
    "ram_available_gb": ram_available_gb,
    "device_type": device_type,
    "gpu_name": gpu_name,
    "vram_total_gb": vram_total_gb,
    "vram_available_gb": vram_available_gb,
}

# ---------- Display ----------
print("=" * 55)
print("  SYSTEM HARDWARE PROFILE")
print("=" * 55)
print(f"  CPU        : {cpu_model}")
print(f"  Cores      : {cpu_cores_physical} physical / {cpu_cores_logical} logical")
print(f"  RAM        : {ram_total_gb} GB total / {ram_available_gb} GB available")
device_line = f"  Device     : {device_type.upper()}"
if gpu_name:
    device_line += f"  ({gpu_name})"
print(device_line)
print(f"  Model VRAM : {vram_total_gb} GB budget / {vram_available_gb} GB free")
print("=" * 55)
if device_type == "cpu":
    print("  ** No GPU detected - models will run on CPU (slower).")
    print("     Quantized GGUF models via llama.cpp recommended.")
elif device_type == "mps":
    print("  Apple Silicon - unified memory shared with OS.")
    print("  Budget set to ~70% of RAM for model loading.")
print("=" * 55)

# --- Top 10 RAM-consuming processes ---
ram_used_gb = round(ram_total_gb - ram_available_gb, 1)
print(f"\n  RAM in use: {ram_used_gb} GB ({round(ram_used_gb/ram_total_gb*100, 1)}% of total)")
print("-" * 55)
print("  TOP 10 RAM-CONSUMING PROCESSES")
print("-" * 55)
procs = []
for p in psutil.process_iter(['pid', 'name', 'memory_info']):
    try:
        mi = p.info.get('memory_info')
        if mi is None:
            continue
        mem_gb = mi.rss / (1024**3)
        if mem_gb > 0.01:
            procs.append((p.info['name'] or '?', p.info['pid'], mem_gb))
    except (psutil.NoSuchProcess, psutil.AccessDenied, psutil.ZombieProcess, AttributeError):
        pass
procs.sort(key=lambda x: x[2], reverse=True)
print(f"  {'Process':<30s} {'PID':>7s} {'RAM (GB)':>10s} {'% of Used':>10s}")
print(f"  {'-'*30} {'-'*7} {'-'*10} {'-'*10}")
for name, pid, mem_gb in procs[:10]:
    pct = round(mem_gb / ram_used_gb * 100, 1) if ram_used_gb > 0 else 0
    print(f"  {name:<30s} {pid:>7d} {mem_gb:>9.2f}  {pct:>8.1f}%")
top_total = sum(m for _, _, m in procs[:10])
print(f"  {'':30s} {'':>7s} {'─'*10} {'─'*10}")
print(f"  {'Top 10 total':<30s} {'':>7s} {top_total:>9.2f}  {round(top_total/ram_used_gb*100,1) if ram_used_gb > 0 else 0:>8.1f}%")
print(f"\n  💡 Close heavy processes above to free RAM and fit larger models.")
print("=" * 55)

  SYSTEM HARDWARE PROFILE
  CPU        : arm
  Cores      : 12 physical / 12 logical
  RAM        : 32.0 GB total / 13.6 GB available
  Device     : MPS  (Apple Silicon (MPS))
  Model VRAM : 22.4 GB budget / 9.5 GB free
  Apple Silicon - unified memory shared with OS.
  Budget set to ~70% of RAM for model loading.

  RAM in use: 18.4 GB (57.5% of total)
-------------------------------------------------------
  TOP 10 RAM-CONSUMING PROCESSES
-------------------------------------------------------
  Process                            PID   RAM (GB)  % of Used
  ------------------------------ ------- ---------- ----------
  Code Helper (Plugin)             87705      2.70      14.7%
  fileproviderd                     1624      1.01       5.5%
  Microsoft Teams WebView Helper (Renderer)   15921      0.85       4.6%
  Code Helper (Renderer)           63329      0.74       4.0%
  Code Helper (Plugin)             63731      0.69       3.8%
  Google Chrome Helper (Renderer)   18886      0.50 

In [8]:
# ============================================================
# CELL 2: Load Models — from JSON file or HuggingFace API
# ============================================================
import re, json, os

# ===================== CHOOSE DATA SOURCE =====================
# Set to "json" to load from local file (fast, 593 models)
# Set to "api"  to fetch live from HuggingFace Hub (slower, ~15-30s)
DATA_SOURCE = "json"

JSON_PATH   = "hf_models.json"   # Path to local JSON file
MODEL_LIMIT = 500                # Only used for API mode
TASK_FILTER = "text-generation"  # Only used for API mode

# ===================== PIPELINE FILTER ========================
# Filter models by pipeline_tag. Use "text-generation" (default),
# or "all" to include every pipeline type in the JSON/API data.
PIPELINE_FILTER = "text-generation"
# ==============================================================

# ----- VRAM estimation constants (bytes per parameter) -----
BYTES_PER_PARAM = {
    "FP32": 4.0, "FP16": 2.0, "BF16": 2.0,
    "INT8": 1.0, "8bit": 1.0,
    "INT4": 0.5, "4bit": 0.5,
    "GPTQ": 0.5, "AWQ": 0.5,
    "GPTQ-Int4": 0.5, "GPTQ-Int8": 1.0,
    "AWQ-4bit": 0.5, "AWQ-8bit": 1.0,
    # GGUF quant levels
    "Q2_K": 0.31, "Q3_K_S": 0.38, "Q3_K_M": 0.41, "Q3_K_L": 0.44,
    "Q4_0": 0.50, "Q4_K_S": 0.53, "Q4_K_M": 0.56,
    "Q5_0": 0.63, "Q5_K_S": 0.66, "Q5_K_M": 0.69,
    "Q6_K": 0.81, "Q8_0": 1.0, "IQ2_XS": 0.28, "IQ3_XS": 0.36,
}
OVERHEAD_FACTOR = 1.15  # 15% overhead for KV cache and runtime activations

# ==============================================================
# OPTION A: Load from local JSON file
# ==============================================================
def load_from_json(path):
    """Parse the local hf_models.json into model_rows."""
    with open(path, "r") as f:
        data = json.load(f)

    rows = []
    skipped = 0
    for m in data:
        # Map format to quant_type
        fmt = (m.get("format") or "").lower()
        quant_raw = m.get("quantization", "")

        if fmt == "gguf":
            qtype = "GGUF"
            qlevel = quant_raw if quant_raw else "Q4_K_M"
        elif fmt == "gptq":
            qtype = "GPTQ"
            qlevel = quant_raw if quant_raw else "4bit"
        elif fmt == "awq":
            qtype = "AWQ"
            qlevel = quant_raw if quant_raw else "4bit"
        elif fmt == "mlx":
            qtype = "MLX"
            qlevel = quant_raw if quant_raw else "4bit"
        else:
            qtype = "Full"
            qlevel = "FP16"

        param_count = m.get("parameters_raw", 0)
        params_b = round(param_count / 1e9, 2) if param_count else None

        # VRAM estimate: use pre-computed min_vram_gb if available, else compute
        est_vram = m.get("min_vram_gb")
        if not est_vram and param_count:
            bpp = BYTES_PER_PARAM.get(qlevel, BYTES_PER_PARAM.get(qtype, 2.0))
            est_vram = round((param_count * bpp / (1024**3)) * OVERHEAD_FACTOR, 2)

        ctx_len = m.get("context_length")
        name = m.get("name", "")

        row = {
            "model_id": name,
            "author": m.get("provider", name.split("/")[0] if "/" in name else ""),
            "downloads": m.get("hf_downloads", 0),
            "likes": m.get("hf_likes", 0),
            "trending": 0,
            "params_b": params_b,
            "quant_type": qtype,
            "quant_level": qlevel,
            "est_vram_gb": est_vram,
            "recommended_ram_gb": m.get("recommended_ram_gb"),
            "context_len": ctx_len,
            "license": None,
            "library": None,
            "gguf_file": None,
            "architecture": m.get("architecture"),
            "use_case": m.get("use_case"),
            "pipeline_tag": m.get("pipeline_tag", "text-generation"),
        }
        if row["est_vram_gb"] is not None or row["params_b"] is not None:
            rows.append(row)
        else:
            skipped += 1

    return rows, len(data), skipped

# ==============================================================
# OPTION B: Fetch from HuggingFace API
# ==============================================================
def load_from_api(limit, task_filter):
    """Fetch models from HuggingFace Hub API and parse into model_rows."""
    from huggingface_hub import HfApi
    api = HfApi()

    print(f"Fetching top {limit} '{task_filter}' models from HuggingFace Hub...")
    print("(This may take 15-30 seconds with expanded metadata)\n")

    raw_models = list(api.list_models(
        filter=task_filter,
        sort="downloads",
        limit=limit,
        expand=["safetensors", "gguf", "config", "tags",
                "cardData", "trendingScore", "siblings"],
    ))

    def _detect_quantization(model):
        tags = [t.lower() for t in (model.tags or [])]
        name_lower = model.modelId.lower()
        siblings = model.siblings or []
        filenames = [s.rfilename for s in siblings]
        param_count = None
        if model.safetensors and hasattr(model.safetensors, "parameter_count"):
            pc = model.safetensors.parameter_count
            if isinstance(pc, dict):
                param_count = sum(pc.values())
            elif isinstance(pc, (int, float)):
                param_count = int(pc)
        results = []
        gguf_files = [f for f in filenames if f.lower().endswith(".gguf")]
        if gguf_files or "gguf" in tags:
            for fname in gguf_files:
                match = re.search(r"((?:IQ|Q)\d[A-Za-z0-9_]*)", fname, re.IGNORECASE)
                qlevel = match.group(1).upper() if match else "Q4_K_M"
                bpp = BYTES_PER_PARAM.get(qlevel, 0.56)
                file_size_gb = None
                for s in siblings:
                    if s.rfilename == fname and hasattr(s, "size") and s.size:
                        file_size_gb = s.size / (1024**3)
                        break
                if file_size_gb:
                    est_vram = round(file_size_gb * OVERHEAD_FACTOR, 2)
                elif param_count:
                    est_vram = round((param_count * bpp / (1024**3)) * OVERHEAD_FACTOR, 2)
                else:
                    est_vram = None
                results.append(("GGUF", qlevel, est_vram, fname))
            if not results and param_count:
                est_vram = round((param_count * 0.56 / (1024**3)) * OVERHEAD_FACTOR, 2)
                results.append(("GGUF", "Q4_K_M", est_vram, None))
            return results, param_count
        if "gptq" in tags or "gptq" in name_lower:
            est_vram = round((param_count * 0.5 / (1024**3)) * OVERHEAD_FACTOR, 2) if param_count else None
            return [("GPTQ", "4bit", est_vram, None)], param_count
        if "awq" in tags or "awq" in name_lower:
            est_vram = round((param_count * 0.5 / (1024**3)) * OVERHEAD_FACTOR, 2) if param_count else None
            return [("AWQ", "4bit", est_vram, None)], param_count
        dtype = "FP16"
        if model.safetensors and hasattr(model.safetensors, "parameter_count"):
            pc = model.safetensors.parameter_count
            if isinstance(pc, dict) and pc:
                dominant = max(pc, key=pc.get).upper()
                dtype_map = {"F32": "FP32", "F16": "FP16", "BF16": "BF16", "I8": "INT8", "I4": "INT4"}
                dtype = dtype_map.get(dominant, dominant)
        bpp = BYTES_PER_PARAM.get(dtype, 2.0)
        est_vram = round((param_count * bpp / (1024**3)) * OVERHEAD_FACTOR, 2) if param_count else None
        return [("Full", dtype, est_vram, None)], param_count

    def _get_context_length(model):
        cfg = model.config or {}
        if isinstance(cfg, dict):
            inner = cfg
            if len(cfg) == 1 and isinstance(list(cfg.values())[0], dict):
                inner = list(cfg.values())[0]
            for key in ["max_position_embeddings", "max_sequence_length",
                         "n_positions", "seq_length", "sliding_window",
                         "max_seq_len", "model_max_length"]:
                val = inner.get(key) or cfg.get(key)
                if val and isinstance(val, (int, float)):
                    return int(val)
        return None

    def _get_license(model):
        for tag in (model.tags or []):
            if tag.startswith("license:"):
                return tag.split(":", 1)[1]
        return None

    rows = []
    skipped = 0
    for m in raw_models:
        quant_variants, param_count = _detect_quantization(m)
        ctx_len = _get_context_length(m)
        lic = _get_license(m)
        params_b = round(param_count / 1e9, 2) if param_count else None
        for (qtype, qlevel, est_vram, gguf_file) in quant_variants:
            row = {
                "model_id": m.modelId,
                "author": m.author,
                "downloads": m.downloads or 0,
                "likes": m.likes or 0,
                "trending": getattr(m, "trending_score", None) or 0,
                "params_b": params_b,
                "quant_type": qtype,
                "quant_level": qlevel,
                "est_vram_gb": est_vram,
                "recommended_ram_gb": None,
                "context_len": ctx_len,
                "license": lic,
                "library": m.library_name,
                "gguf_file": gguf_file,
                "architecture": None,
                "use_case": None,
                "pipeline_tag": "text-generation",
            }
            if row["est_vram_gb"] is not None or row["params_b"] is not None:
                rows.append(row)
            else:
                skipped += 1

    return rows, len(raw_models), skipped

# ==============================================================
# Load data based on selected source
# ==============================================================
if DATA_SOURCE == "json":
    print(f"Loading models from local JSON file: {JSON_PATH}")
    model_rows, total_repos, skipped = load_from_json(JSON_PATH)
    print(f"Source: Local JSON ({total_repos} models in file)")
elif DATA_SOURCE == "api":
    model_rows, total_repos, skipped = load_from_api(MODEL_LIMIT, TASK_FILTER)
    print(f"Source: HuggingFace API (fetched {total_repos} repos)")
else:
    raise ValueError(f"Invalid DATA_SOURCE: {DATA_SOURCE!r}. Use 'json' or 'api'.")

# Filter by pipeline_tag
pre_filter = len(model_rows)
if PIPELINE_FILTER.lower() != "all":
    model_rows = [r for r in model_rows if r.get("pipeline_tag", "text-generation") == PIPELINE_FILTER]
filtered_out = pre_filter - len(model_rows)

pipeline_label = PIPELINE_FILTER if PIPELINE_FILTER.lower() != "all" else "all pipelines"
print(f"\nParsed {len(model_rows)} {pipeline_label} model variants from {total_repos} repos.")
if filtered_out:
    print(f"  ({filtered_out} non-{PIPELINE_FILTER} models filtered out)")
if skipped:
    print(f"  ({skipped} variants skipped - insufficient metadata)")

# Breakdown by quant type
for qt in ["Full", "GGUF", "GPTQ", "AWQ", "MLX"]:
    count = sum(1 for r in model_rows if r["quant_type"] == qt)
    if count:
        print(f"  {qt:15s}: {count}")
print(f"  {'TOTAL':15s}: {len(model_rows)}")

Loading models from local JSON file: hf_models.json
Source: Local JSON (593 models in file)

Parsed 559 text-generation model variants from 593 repos.
  (34 non-text-generation models filtered out)
  Full           : 29
  GGUF           : 404
  GPTQ           : 18
  AWQ            : 45
  MLX            : 63
  TOTAL          : 559


In [9]:
# ============================================================
# CELL 3: Scoring Engine
# ============================================================
import math
import numpy as np

# ---- Scoring Weights (must sum to 1.0) ----
W_FIT     = 0.40
W_QUALITY = 0.25
W_SPEED   = 0.25
W_CONTEXT = 0.10

# Pre-compute dataset statistics for percentile-based scoring
all_downloads = [r["downloads"] for r in model_rows if r["downloads"] > 0]
all_likes     = [r["likes"] for r in model_rows if r["likes"] > 0]
all_params    = [r["params_b"] for r in model_rows if r["params_b"]]

log_downloads = np.log1p(all_downloads) if all_downloads else np.array([0])
log_likes     = np.log1p(all_likes) if all_likes else np.array([0])

dl_p90 = float(np.percentile(log_downloads, 90)) if len(log_downloads) > 1 else 1.0
lk_p90 = float(np.percentile(log_likes, 90)) if len(log_likes) > 1 else 1.0
max_params = max(all_params) if all_params else 70.0

vram_budget = HARDWARE["vram_available_gb"]
device = HARDWARE["device_type"]


def score_quality(row):
    """0-100: Community trust signal based on downloads + likes + trending."""
    dl_score = min(math.log1p(row["downloads"]) / dl_p90, 1.0) * 60
    lk_score = min(math.log1p(row["likes"]) / lk_p90, 1.0) * 30
    tr_score = min(row["trending"] / 50, 1.0) * 10 if row["trending"] else 0
    return round(dl_score + lk_score + tr_score, 1)


def score_speed(row):
    """0-100: Inversely proportional to model size relative to hardware.
    Smaller models = faster. Quantized models get a bonus."""
    params = row["params_b"]
    if not params:
        return 30.0  # unknown size, conservative score

    # Base: how small is this relative to the largest model?
    size_ratio = params / max_params
    base_speed = max(0, (1 - size_ratio) * 80)

    # Quantization bonus: quantized models are faster at same param count
    quant_bonus = 0
    if row["quant_type"] in ("GGUF", "GPTQ", "AWQ"):
        quant_bonus = 15
    elif row["quant_level"] in ("INT8", "8bit"):
        quant_bonus = 8

    # Device bonus: GPU is much faster than CPU for same model
    device_bonus = 0
    if device == "cuda":
        device_bonus = 5
    elif device == "mps":
        device_bonus = 3

    return round(min(base_speed + quant_bonus + device_bonus, 100), 1)


def score_fit(row):
    """0-100: Will this model fit in available VRAM/RAM?
    Core metric - a model you can't load is useless."""
    est_vram = row["est_vram_gb"]
    if est_vram is None:
        return 20.0  # unknown, pessimistic

    if vram_budget <= 0:
        return 5.0

    usage_ratio = est_vram / vram_budget

    if usage_ratio <= 0.50:
        score = 100.0   # Fits with plenty of room
    elif usage_ratio <= 0.70:
        score = 90.0    # Comfortable fit
    elif usage_ratio <= 0.85:
        score = 75.0    # Moderate fit
    elif usage_ratio <= 0.95:
        score = 55.0    # Tight fit - may work
    elif usage_ratio <= 1.0:
        score = 35.0    # Very tight - might OOM
    elif usage_ratio <= 1.2:
        score = 15.0    # Likely won't load
    else:
        score = 0.0     # Definitely won't fit

    # Additional penalty for CPU-only inference on large models
    if device == "cpu" and row.get("params_b") and row["params_b"] > 13:
        score = max(score - 20, 0)

    return score


def score_context(row):
    """0-100: Normalized context window score. Larger context = more useful."""
    ctx = row["context_len"]
    if not ctx:
        return 30.0  # unknown

    # Log-scale scoring with diminishing returns
    if ctx <= 2048:
        return 20.0
    elif ctx <= 4096:
        return 40.0
    elif ctx <= 8192:
        return 60.0
    elif ctx <= 16384:
        return 70.0
    elif ctx <= 32768:
        return 80.0
    elif ctx <= 65536:
        return 87.0
    elif ctx <= 131072:
        return 93.0
    else:
        return 100.0


def get_verdict(fit_score):
    """Human-readable verdict based on fit score."""
    if fit_score >= 85:
        return "Excellent"
    elif fit_score >= 60:
        return "Good"
    elif fit_score >= 30:
        return "Marginal"
    else:
        return "Won't Run"


# ----- Apply scoring to all model rows -----
for row in model_rows:
    row["s_quality"] = score_quality(row)
    row["s_speed"]   = score_speed(row)
    row["s_fit"]     = score_fit(row)
    row["s_context"] = score_context(row)
    row["composite"] = round(
        W_FIT * row["s_fit"] +
        W_QUALITY * row["s_quality"] +
        W_SPEED * row["s_speed"] +
        W_CONTEXT * row["s_context"], 1
    )
    row["verdict"] = get_verdict(row["s_fit"])

# Sort by composite score descending
model_rows.sort(key=lambda r: r["composite"], reverse=True)

print(f"Scoring complete for {len(model_rows)} model variants.")
print(f"  VRAM budget: {vram_budget} GB ({device.upper()})")
print(f"  Weights: Fit={W_FIT}, Quality={W_QUALITY}, Speed={W_SPEED}, Context={W_CONTEXT}")
verdicts = {}
for r in model_rows:
    verdicts[r["verdict"]] = verdicts.get(r["verdict"], 0) + 1
for v in ["Excellent", "Good", "Marginal", "Won't Run"]:
    print(f"  {v}: {verdicts.get(v, 0)}")

Scoring complete for 559 model variants.
  VRAM budget: 9.5 GB (MPS)
  Weights: Fit=0.4, Quality=0.25, Speed=0.25, Context=0.1
  Excellent: 338
  Good: 37
  Marginal: 11
  Won't Run: 173


In [10]:
# ============================================================
# CELL 4: Interactive Results Table
# ============================================================
import pandas as pd

# -------------------- FILTERS (adjust as needed) --------------------
MIN_PARAMS_B    = 0       # Minimum parameter count in billions
MAX_PARAMS_B    = 999     # Maximum parameter count in billions
MIN_CONTEXT     = 0       # Minimum context window length
QUANT_FILTER    = "all"   # "all" | "full" | "quantized" | "gguf" | "gptq" | "awq"
TOP_N           = 50      # Max rows to display
# --------------------------------------------------------------------

# Apply filters
filtered = model_rows.copy()

if MIN_PARAMS_B > 0:
    filtered = [r for r in filtered if r["params_b"] and r["params_b"] >= MIN_PARAMS_B]
if MAX_PARAMS_B < 999:
    filtered = [r for r in filtered if r["params_b"] and r["params_b"] <= MAX_PARAMS_B]
if MIN_CONTEXT > 0:
    filtered = [r for r in filtered if r["context_len"] and r["context_len"] >= MIN_CONTEXT]

qf = QUANT_FILTER.lower()
if qf == "full":
    filtered = [r for r in filtered if r["quant_type"] == "Full"]
elif qf == "quantized":
    filtered = [r for r in filtered if r["quant_type"] != "Full"]
elif qf in ("gguf", "gptq", "awq"):
    filtered = [r for r in filtered if r["quant_type"].lower() == qf]

filtered = filtered[:TOP_N]

# Build DataFrame
df = pd.DataFrame(filtered)

display_cols = [
    "model_id", "quant_type", "quant_level", "params_b", "est_vram_gb",
    "context_len", "downloads", "s_quality", "s_speed", "s_fit",
    "s_context", "composite", "verdict"
]
# Only show columns that exist
display_cols = [c for c in display_cols if c in df.columns]
df_display = df[display_cols].copy()

# Rename columns for readability
df_display.columns = [
    "Model", "Type", "Quant", "Params(B)", "VRAM(GB)",
    "Context", "Downloads", "Quality", "Speed", "Fit",
    "CtxScore", "Score", "Verdict"
][:len(display_cols)]


def color_verdict(val):
    colors = {
        "Excellent": "background-color: #2d6a2d; color: white",
        "Good":      "background-color: #4a8c3f; color: white",
        "Marginal":  "background-color: #c88a2c; color: white",
        "Won't Run": "background-color: #a83232; color: white",
    }
    return colors.get(val, "")


def color_score(val):
    try:
        v = float(val)
    except (ValueError, TypeError):
        return ""
    if v >= 80:
        return "background-color: #2d6a2d; color: white"
    elif v >= 60:
        return "background-color: #4a8c3f; color: white"
    elif v >= 40:
        return "background-color: #c88a2c; color: white"
    else:
        return "background-color: #a83232; color: white"


# Style and display
styled = (
    df_display.style
    .map(color_verdict, subset=["Verdict"])
    .map(color_score, subset=["Score"])
    .format({
        "Params(B)": "{:.1f}",
        "VRAM(GB)": "{:.1f}",
        "Context": lambda x: f"{x:,.0f}" if pd.notna(x) else "?",
        "Downloads": lambda x: f"{x:,.0f}",
        "Quality": "{:.0f}",
        "Speed": "{:.0f}",
        "Fit": "{:.0f}",
        "CtxScore": "{:.0f}",
        "Score": "{:.1f}",
    }, na_rep="?")
    .set_properties(**{"font-size": "11px"})
    .set_table_styles([
        {"selector": "th", "props": [("font-size", "11px"), ("text-align", "center")]},
    ])
    .hide(axis="index")
)

print(f"Showing {len(df_display)} models (filtered from {len(model_rows)} total)")
print(f"Filters: params={MIN_PARAMS_B}-{MAX_PARAMS_B}B, context>={MIN_CONTEXT}, quant={QUANT_FILTER}\n")
styled

Showing 50 models (filtered from 559 total)
Filters: params=0-999B, context>=0, quant=all



Model,Type,Quant,Params(B),VRAM(GB),Context,Downloads,Quality,Speed,Fit,CtxScore,Score,Verdict
Qwen/Qwen3-4B-Instruct-2507,GGUF,Q4_K_M,4.0,2.1,"262,144","5,604,976",89,98,100,100,96.7,Excellent
Nanbeige/Nanbeige4.1-3B,GGUF,Q4_K_M,3.9,2.0,"262,144","734,323",89,98,100,100,96.6,Excellent
Qwen/Qwen3-4B-Thinking-2507,GGUF,Q4_K_M,4.0,2.1,"262,144","1,056,750",88,98,100,100,96.3,Excellent
deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B,GGUF,Q4_K_M,1.8,0.9,"131,072","641,594",88,98,100,93,95.8,Excellent
Qwen/Qwen3-0.6B,GGUF,Q4_K_M,0.8,0.5,"40,960","13,320,335",90,98,100,87,95.7,Excellent
Qwen/Qwen3-8B,GGUF,Q4_K_M,8.2,4.2,"40,960","8,916,125",90,97,100,87,95.5,Excellent
HuggingFaceTB/SmolLM3-3B,GGUF,Q4_K_M,3.1,1.6,"65,536","838,923",89,98,100,87,95.4,Excellent
deepseek-ai/DeepSeek-R1-Distill-Qwen-7B,GGUF,Q4_K_M,7.6,3.9,"131,072","576,713",87,97,100,93,95.4,Excellent
mistralai/Mistral-7B-Instruct-v0.2,GGUF,Q4_K_M,7.2,3.7,"32,768","3,048,605",90,98,100,80,94.9,Excellent
Qwen/Qwen3-1.7B,GGUF,Q4_K_M,2.0,1.0,"40,960","7,069,931",86,98,100,87,94.8,Excellent


In [11]:
# ============================================================
# CELL 5: Top Recommendations & Hardware Insight
# ============================================================
from IPython.display import display, Markdown

def recommend(rows, label, filter_fn):
    """Pick the best model matching filter_fn, return formatted string."""
    candidates = [r for r in rows if filter_fn(r) and r["verdict"] != "Won't Run"]
    if not candidates:
        return f"**{label}:** No suitable models found.\n"
    best = candidates[0]  # Already sorted by composite
    vram_str = f"{best['est_vram_gb']:.1f} GB" if best["est_vram_gb"] else "unknown"
    ctx_str = f"{best['context_len']:,}" if best["context_len"] else "unknown"
    backend = "llama.cpp" if best["quant_type"] == "GGUF" else "transformers"
    if best["quant_type"] in ("GPTQ", "AWQ"):
        backend = f"transformers + auto-{best['quant_type'].lower()}"
    return (
        f"**{label}:** `{best['model_id']}`\n"
        f"  - Type: {best['quant_type']} {best['quant_level']} | "
        f"Params: {best['params_b']}B | VRAM: {vram_str} | Context: {ctx_str}\n"
        f"  - Score: {best['composite']:.1f}/100 "
        f"(Fit:{best['s_fit']:.0f} Qual:{best['s_quality']:.0f} "
        f"Spd:{best['s_speed']:.0f} Ctx:{best['s_context']:.0f})\n"
        f"  - Backend: {backend}\n"
    )


# Generate recommendations
recs = []
recs.append(recommend(model_rows, "Best Overall",
                       lambda r: True))
recs.append(recommend(model_rows, "Best Full-Precision",
                       lambda r: r["quant_type"] == "Full"))
recs.append(recommend(model_rows, "Best GGUF (Quantized)",
                       lambda r: r["quant_type"] == "GGUF"))
recs.append(recommend(model_rows, "Best GPTQ/AWQ",
                       lambda r: r["quant_type"] in ("GPTQ", "AWQ")))

# Hardware utilization insight
vram = HARDWARE["vram_available_gb"]
dev = HARDWARE["device_type"].upper()

# Estimate what model sizes fit at different quant levels
fp16_max_b = round(vram / (2.0 * OVERHEAD_FACTOR), 1)
q4_max_b   = round(vram / (0.56 * OVERHEAD_FACTOR), 1)
q8_max_b   = round(vram / (1.0 * OVERHEAD_FACTOR), 1)

insight = (
    f"## Hardware Utilization Insight\n\n"
    f"With **{vram:.1f} GB** available on **{dev}**, your system can approximately run:\n\n"
    f"| Format | Max Model Size | Example |\n"
    f"|--------|---------------|----------|\n"
    f"| FP16 (full) | ~{fp16_max_b}B params | "
    f"{'Llama-3-8B' if fp16_max_b >= 8 else 'Phi-3-mini (3.8B)' if fp16_max_b >= 4 else 'TinyLlama (1.1B)'} |\n"
    f"| GGUF Q8 | ~{q8_max_b}B params | "
    f"{'Llama-3-70B' if q8_max_b >= 70 else 'Llama-3-8B' if q8_max_b >= 8 else 'Phi-3 (3.8B)'} |\n"
    f"| GGUF Q4_K_M | ~{q4_max_b}B params | "
    f"{'Llama-3-70B' if q4_max_b >= 70 else 'Qwen2-32B' if q4_max_b >= 32 else 'Llama-3-8B' if q4_max_b >= 8 else 'Phi-3 (3.8B)'} |\n"
)

if HARDWARE["device_type"] == "cpu":
    insight += (
        "\n> **CPU-only note:** While models may *fit* in RAM, inference will be slow. "
        "Recommend GGUF Q4 models under 7B via `llama-cpp-python` for usable speeds.\n"
    )
elif HARDWARE["device_type"] == "mps":
    insight += (
        "\n> **Apple Silicon note:** Unified memory is shared with the OS. "
        "Actual usable VRAM depends on what else is running. "
        "Both `transformers` (MPS backend) and `llama.cpp` (Metal) work well.\n"
    )

# Display
md = "## Top Recommendations\n\n" + "\n".join(recs) + "\n---\n\n" + insight
display(Markdown(md))

# Quick-start commands
print("\n" + "=" * 55)
print("  QUICK-START COMMANDS")
print("=" * 55)
best_overall = next((r for r in model_rows if r["verdict"] != "Won't Run"), None)
if best_overall:
    if best_overall["quant_type"] == "GGUF":
        print(f"\n  # Install llama-cpp-python (with Metal/CUDA support):")
        print(f"  pip install llama-cpp-python")
        print(f"\n  # Download & run:")
        print(f"  # huggingface-cli download {best_overall['model_id']}")
        if best_overall.get("gguf_file"):
            print(f"  # File: {best_overall['gguf_file']}")
    else:
        print(f"\n  # Install transformers:")
        print(f"  pip install transformers torch accelerate")
        print(f"\n  # Load model:")
        print(f"  from transformers import AutoModelForCausalLM, AutoTokenizer")
        print(f"  model = AutoModelForCausalLM.from_pretrained('{best_overall['model_id']}')")
print("=" * 55)

## Top Recommendations

**Best Overall:** `Qwen/Qwen3-4B-Instruct-2507`
  - Type: GGUF Q4_K_M | Params: 4.02B | VRAM: 2.1 GB | Context: 262,144
  - Score: 96.7/100 (Fit:100 Qual:89 Spd:98 Ctx:100)
  - Backend: llama.cpp

**Best Full-Precision:** `LiquidAI/LFM2.5-1.2B-Base`
  - Type: Full FP16 | Params: 1.17B | VRAM: 0.6 GB | Context: 128,000
  - Score: 70.0/100 (Fit:100 Qual:0 Spd:83 Ctx:93)
  - Backend: transformers

**Best GGUF (Quantized):** `Qwen/Qwen3-4B-Instruct-2507`
  - Type: GGUF Q4_K_M | Params: 4.02B | VRAM: 2.1 GB | Context: 262,144
  - Score: 96.7/100 (Fit:100 Qual:89 Spd:98 Ctx:100)
  - Backend: llama.cpp

**Best GPTQ/AWQ:** `cyankiwi/GLM-4.7-Flash-AWQ-4bit`
  - Type: AWQ AWQ-4bit | Params: 6.41B | VRAM: 3.3 GB | Context: 202,752
  - Score: 92.0/100 (Fit:100 Qual:71 Spd:98 Ctx:100)
  - Backend: transformers + auto-awq

---

## Hardware Utilization Insight

With **9.5 GB** available on **MPS**, your system can approximately run:

| Format | Max Model Size | Example |
|--------|---------------|----------|
| FP16 (full) | ~4.1B params | Phi-3-mini (3.8B) |
| GGUF Q8 | ~8.3B params | Llama-3-8B |
| GGUF Q4_K_M | ~14.8B params | Llama-3-8B |

> **Apple Silicon note:** Unified memory is shared with the OS. Actual usable VRAM depends on what else is running. Both `transformers` (MPS backend) and `llama.cpp` (Metal) work well.



  QUICK-START COMMANDS

  # Install llama-cpp-python (with Metal/CUDA support):
  pip install llama-cpp-python

  # Download & run:
  # huggingface-cli download Qwen/Qwen3-4B-Instruct-2507
